# 🎬 VidGenie: Text-to-Video AI Generator

Generate short-form captioned videos automatically from a text prompt using:
- **LLM (Groq / OpenAI):** Generates engaging, concise video scripts.
- **Edge-TTS:** Converts text to speech with natural voices.
- **Whisper:** Generates accurate, word-level timed captions.
- **Pexels API:** Queries and downloads matching HD background videos.
- **MoviePy:** Renders and stitches audio, footage, and styled captions into a final MP4.

## 1. Setup Environment & Repository

In [ ]:
# Clone repository (if running in a fresh Colab environment)
!git clone https://github.com/lekhan-03/VidGenieAI.git
%cd VidGenieAI

## 2. Install System Tools & Python Dependencies

MoviePy requires `imagemagick` and `ffmpeg` installed with relaxed path security policies for font/caption rendering.

In [ ]:
# Install ImageMagick & configure policy for text rendering
!apt-get update -y > /dev/null
!apt-get install -y imagemagick ffmpeg > /dev/null
!sed -i '/<policy domain="path" rights="none" pattern="@\\*"/d' /etc/ImageMagick-6/policy.xml

# Install Python dependencies
!pip install -r requirements.txt
!pip install groq edge-tts whisper-timestamped openai-whisper streamlit pyngrok

## 3. Configure API Keys (`.env`)

Provide your API keys below:
- **Groq API Key:** Get free from console.groq.com
- **Pexels API Key:** Get free from pexels.com/api

In [ ]:
%%writefile .env
# --- AI Provider Selection ---
LLM_PROVIDER=groq
GROQ_MODEL=openai/gpt-oss-20b
GROQ_API_KEY=YOUR_GROQ_API_KEY_HERE

# --- Visuals ---
PEXELS_API_KEY=YOUR_PEXELS_API_KEY_HERE
VIDEO_ORIENTATION=portrait

# --- Audio & Captions ---
STT_PROVIDER=whisper
TTS_PROVIDER=edgetts
EDGETTS_VOICE=en-US-GuyNeural

# --- Caption Styling ---
CAPTIONS_ENABLED=true
CAPTION_FONT_SIZE=90
CAPTION_FONT_COLOR=white
CAPTION_FONT_FACE=Arial-Bold
CAPTION_STROKE_WIDTH=3
CAPTION_STROKE_COLOR=black
CAPTION_POSITION=bottom_center

# --- Rendering Engine ---
RENDER_ENGINE=moviepy

## 4. Run the Pipeline Programmatically

In [ ]:
import asyncio
import os
from utility.script.script_generator import generate_script
from utility.audio.audio_generator import generate_audio
from utility.captions.timed_captions_generator import generate_timed_captions
from utility.video.background_video_generator import generate_video_url
from utility.render.render_engine import get_output_media
from utility.video.video_search_query_generator import getVideoSearchQueriesTimed, merge_empty_intervals

async def create_video(topic: str, voice: str = "en-US-GuyNeural"):
    audio_file = "audio_tts.wav"
    video_server = "pexel"
    
    print(f"\ud83d\udccc Topic: {topic}")
    print("1\ufe0f\u20e3 Generating Script...")
    script = generate_script(topic)
    print(f"\nScript:\n{script}\n")
    
    print(f"2\ufe0f\u20e3 Generating Speech Audio with voice '{voice}'...")
    await generate_audio(script, audio_file, voice=voice)
    
    print("3\ufe0f\u20e3 Transcribing & Syncing Timed Captions...")
    timed_captions = generate_timed_captions(audio_file)
    
    print("4\ufe0f\u20e3 Querying Background Videos from Pexels...")
    search_terms = getVideoSearchQueriesTimed(script, timed_captions)
    video_urls = generate_video_url(search_terms, video_server)
    video_urls = merge_empty_intervals(video_urls)
    
    print("5\ufe0f\u20e3 Rendering Final Video...")
    output_path = get_output_media(audio_file, timed_captions, video_urls, video_server)
    print(f"\n\u2705 Done! Video saved at: {output_path}")
    return output_path

# Run for an example topic
final_video_path = await create_video(topic="Fascinating Facts About Deep Oceans", voice="en-US-GuyNeural")

## 5. Preview Rendered Video in Notebook

In [ ]:
from IPython.display import HTML
from base64 import b64encode

def show_video(file_path):
    with open(file_path, "rb") as f:
        video_encoded = b64encode(f.read()).decode()
    return HTML(f'''
    <video width="360" height="640" controls autoplay loop>
        <source src="data:video/mp4;base64,{video_encoded}" type="video/mp4">
    </video>
    ''')

show_video("rendered_video.mp4")